In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

PROJECT_ROOT = Path(os.path.abspath('..'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']

DB_PATH = str(PROJECT_ROOT / 'data' / 'processed' / 'energy_sector.db')
FIG_DIR = str(PROJECT_ROOT / 'docs' / 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print(f"Conectando ao banco: {DB_PATH}")

In [ ]:
# 1. Consulta SQL direta ao SQLite
conn = sqlite3.connect(DB_PATH)
query = """
    SELECT f.data, f.id_submercado, d.nom_submercado, d.regiao,
           f.ear_percentual, f.ear_mwmes, f.ena_mwmed, f.ena_percentual_mlt,
           f.pld_medio, f.pld_min, f.pld_max, f.ano, f.mes, f.dia
    FROM fato_energia_daily f
    JOIN dim_submercado d ON f.id_submercado = d.id_submercado
    ORDER BY f.data, f.id_submercado;
"""
df = pd.read_sql(query, conn, parse_dates=['data'])
conn.close()

print(f"-> Linhas carregadas: {len(df):,}.")
print(f"-> Período: {df['data'].min().strftime('%d/%m/%Y')} a {df['data'].max().strftime('%d/%m/%Y')}.")
display(df.head(4))

In [ ]:
# 2. Estatística Descritiva por Submercado
desc_stats = df.groupby(['id_submercado', 'nom_submercado']).agg(
    ear_medio=('ear_percentual', 'mean'),
    ear_min=('ear_percentual', 'min'),
    ear_max=('ear_percentual', 'max'),
    ena_mlt_media=('ena_percentual_mlt', 'mean'),
    pld_medio_geral=('pld_medio', 'mean'),
    pld_desvio=('pld_medio', 'std'),
    pld_maximo=('pld_max', 'max')
).round(2)

display(desc_stats)

In [ ]:
# 3. Matriz de Correlações (Pearson vs. Spearman)
from src.analysis import compute_correlations
df_corr = compute_correlations(df)
display(df_corr)

In [ ]:
# 4. Exportação das Figuras (300 DPI)
from src.analysis import (
    export_heatmaps_figure,
    export_dual_axis_series,
    export_scatter_relationship,
    export_scatter_ena_pld
)

export_heatmaps_figure(df, os.path.join(FIG_DIR, 'figura_matriz_correlacao_submercados.png'))
export_dual_axis_series(df, os.path.join(FIG_DIR, 'figura_serie_historica_ear_pld_se_co.png'), submarket='SE/CO')
export_scatter_relationship(df, os.path.join(FIG_DIR, 'figura_dispersao_ear_pld.png'))
export_scatter_ena_pld(df, os.path.join(FIG_DIR, 'figura_dispersao_ena_pld.png'))
print("Figuras geradas com sucesso em docs/figures/!")